# Wildfire GRPO Training (Hugging Face)

Use this notebook only for GRPO training on the Hugging Face GPU runtime.

Cell order:
1. Run Cell 1 first each session (loads env vars + BASE).
2. Run Cell 2 to clone/update and install dependencies.
3. Run Cell 3 for OpenEnv/training preflight.
4. Run Cell 4 for the GRPO training run.

Cell 4 documents two configurations:

- The **default** 60-iteration configuration in `train_grpo.py` (the
  environment was tuned for this; long run on A10G if you use `python train_grpo.py`).
- The **`deadline_v2_a10g`** configuration used for the **hackathon submission**:
  **20 GRPO iterations**, `group_size=2`, `seeds_per_iter=1`, narrower
  `task_curriculum` (hard from iter 12-19). The writeup in `Blog.MD` and
  `README.md` describe this 20-iter run. The final adapter is written under
  `grpo_wildfire/final_adapter` when the run finishes (or when you run the
  optional copy cell if your workflow saves there manually).

After training finishes, use `notebooks/wildfire_http_eval_hf.ipynb` for the
OpenEnv baseline/trained evaluation and final artifacts.

In [ ]:
# Cell 1 (run first each session): env vars + constants (HF runtime)
import os

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
HF_TOKEN = os.environ.get("HF_TOKEN")

if not GITHUB_TOKEN:
    raise RuntimeError("Set GITHUB_TOKEN in the Hugging Face Space/Notebook secrets.")
if not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN in the Hugging Face Space/Notebook secrets.")

os.environ["HF_TOKEN"] = HF_TOKEN

REPO = "jhawaritvik/Scaler-hackathon"
REPO_URL = f"https://github.com/{REPO}.git"
BASE = "/home/user/app/wildfire-env"

print("Configured BASE:", BASE)
print("Tokens available: GITHUB_TOKEN + HF_TOKEN")

In [ ]:
# Cell 2: clone/update + install deps (safe to re-run)
import os
import glob
import subprocess
import sys

if not os.path.exists(BASE):
    subprocess.run(["git", "clone", f"https://{GITHUB_TOKEN}@github.com/{REPO}.git", BASE], check=True)

os.chdir(BASE)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", "main"], check=True)
subprocess.run(["git", "pull", "--rebase", "origin", "main"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[train]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "unsloth", "unsloth-zoo", "bitsandbytes", "xgrammar", "transformers", "trl"], check=True)

nvidia_lib_dirs = glob.glob("/usr/local/lib/python*/dist-packages/nvidia/*/lib")
if nvidia_lib_dirs:
    os.environ["LD_LIBRARY_PATH"] = ":".join(nvidia_lib_dirs) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
    print("LD_LIBRARY_PATH updated with NVIDIA libs")

print("Install complete. cwd:", os.getcwd())

In [ ]:
# Cell 3: preflight for training stack
import subprocess
import sys

subprocess.run(["openenv", "validate", "."], check=True)
subprocess.run([sys.executable, "smoke_test.py"], check=True)
print("Preflight passed")

In [ ]:
# Cell 4: GRPO training run (resume-safe).
#
# Two configurations are documented here:
#
#   (a) "default" â€” `python train_grpo.py` runs the full 60-iteration
#       configuration the environment was tuned for: curriculum easy 0-10,
#       medium 10-25, hard 25-60, group_size=4, seeds_per_iter=2. This is
#       the long "production" training path in the repo.
#
#   (b) "deadline_v2_a10g" â€” the 20-iteration run used for the hackathon
#       submission: group_size=2, seeds_per_iter=1, narrower curriculum
#       (hard iters 12-19). Documented in Blog.MD / README as the submitted
#       training budget. Smaller and faster per wall-clock than (a) on a
#       single A10G.
#
# Pick (a) for a long run. Pick (b) to reproduce the submitted 20-iter run.

import subprocess
import sys
import os

# Project root — train_grpo.py lives here. Resolves on HF Space + locally.
_HERE = os.path.dirname(os.path.abspath(globals().get("__file__", ".")))
_PROJECT_ROOT = os.path.abspath(os.path.join(_HERE, ".."))
if not os.path.isfile(os.path.join(_PROJECT_ROOT, "train_grpo.py")):
    _PROJECT_ROOT = os.path.expanduser("~/app/wildfire-env")

# (a) Default configuration — uncomment to run the full 60-iter training:
# subprocess.run([sys.executable, "train_grpo.py"], cwd=_PROJECT_ROOT, check=True)

# (b) deadline_v2_a10g — 20-iter submission run:
subprocess.run([
    sys.executable,
    "-c",
    f"""
import sys
sys.path.insert(0, {_PROJECT_ROOT!r})
from train_grpo import Config, train

train(Config(
    total_iterations=20,
    task_curriculum=(("easy", 0, 5), ("medium", 5, 12), ("hard", 12, 20)),
    group_size=2,
    seeds_per_iter=1,
    max_episode_steps=20,
    lora_dropout=0.0,
    warmup_iters=3,
    save_every=6,
    run_name="deadline_v2_a10g",
))
""",
    cwd=_PROJECT_ROOT,
], check=True)
print("20-iter deadline_v2_a10g run finished. See grpo_wildfire/log.jsonl, "
      "grpo_wildfire/final_adapter, grpo_wildfire/latest (snapshots + resume state).")

In [ ]:
# Stop Here

Training is complete after Cell 4. Run HTTP evaluation and final artifact generation from `notebooks/wildfire_http_eval_hf.ipynb`.